# Football Prediction System - Ensemble Predictions

This notebook demonstrates team comparison, match predictions, and profit simulation.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import os
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.data_loader import load_league_data, LEAGUE_CONFIG
from src.feature_engineering import engineer_features, MARKET_FEATURES, MARKET_TARGETS
from src.market_models import (
    MarketModel, predict_match, compare_teams, print_team_comparison,
    walk_forward_backtest, EnsembleModel
)
from src.evaluation import simulate_profit, simulate_value_betting, analyze_model_edge

print("Libraries loaded!")

## 1. Load Trained Models

In [ ]:
# Load models for a specific league
LEAGUE = 'epl'
MODELS_DIR = '../models'

# Markets we trained
MARKETS = [
    'match_result', 'over_under_25', 'btts',
    'double_chance_1x', 'double_chance_x2',
    'goals_bucket', 'ht_result'
]

# Load models (try best from ensemble, fallback to xgboost)
models = {}
for market in MARKETS:
    # Try ensemble models first
    for mt in ['random_forest', 'xgboost', 'gradient_boosting']:
        path = os.path.join(MODELS_DIR, LEAGUE, f"{market}_{mt}.pkl")
        if os.path.exists(path):
            models[market] = MarketModel.load(path)
            print(f"✓ Loaded {market} ({mt})")
            break
    else:
        print(f"✗ {market} model not found")

print(f"\nLoaded {len(models)} models for {LEAGUE_CONFIG[LEAGUE]['name']}")

## 2. Load Historical Data (for feature calculation)

In [ ]:
# Load historical data for feature calculation
df_raw = load_league_data(LEAGUE)
df, elos = engineer_features(df_raw, include_elo=True, include_rolling=True,
                              include_odds=True, include_targets=True)

print(f"Loaded {len(df)} historical matches")

## 3. Team Comparison

In [ ]:
# Compare two teams
comparison = compare_teams('Arsenal', 'Chelsea', df)
print_team_comparison(comparison)

In [ ]:
# Another comparison
comparison2 = compare_teams('Man City', 'Liverpool', df)
print_team_comparison(comparison2)

## 4. Make Predictions

In [ ]:
# Example match prediction
home_team = 'Arsenal'
away_team = 'Chelsea'

# Optional: provide bookmaker odds
odds = {
    'home': 1.85,
    'draw': 3.60,
    'away': 4.20,
    'over_25': 1.75,
    'under_25': 2.10,
    'ah_home': 1.90,
    'ah_away': 1.95
}

# Make prediction
prediction = predict_match(home_team, away_team, models, df, odds)

# Pretty print
print(f"\n{'='*60}")
print(f"MATCH PREDICTION: {prediction['match']}")
print(f"{'='*60}")

for market, pred in prediction['predictions'].items():
    print(f"\n{market.upper().replace('_', ' ')}:")
    if 'error' in pred:
        print(f"  Error: {pred['error']}")
    else:
        print(f"  Prediction: {pred['prediction']}")
        if 'confidence' in pred and pred['confidence'] is not None:
            print(f"  Confidence: {pred['confidence']:.1%}")
        if 'probabilities' in pred:
            print(f"  Probabilities:")
            for outcome, prob in pred['probabilities'].items():
                print(f"    {outcome}: {prob:.1%}")

## 5. Batch Predictions

In [ ]:
# Predict multiple matches
matches = [
    ('Arsenal', 'Chelsea'),
    ('Man City', 'Liverpool'),
    ('Man United', 'Tottenham'),
]

all_predictions = []

for home, away in matches:
    pred = predict_match(home, away, models, df)
    all_predictions.append(pred)

# Display as DataFrame
rows = []
for pred in all_predictions:
    row = {'Match': pred['match']}
    for market, p in pred['predictions'].items():
        if 'prediction' in p:
            row[market] = p['prediction']
        if 'confidence' in p and p['confidence'] is not None:
            row[f'{market}_conf'] = f"{p['confidence']:.1%}"
    rows.append(row)

predictions_df = pd.DataFrame(rows)
print(predictions_df.to_string(index=False))

## 6. Profit Simulation on Historical Data

In [ ]:
# Simulate betting on historical data
# Use last 20% as test set
split_idx = int(len(df) * 0.8)
test_df = df.iloc[split_idx:]

# For each market, simulate profit
profit_summary = []

for market_name, model in models.items():
    try:
        features = model.feature_names
        target = MARKET_TARGETS[market_name]
        
        if target not in test_df.columns:
            continue
        
        # Get test data
        test_clean = test_df.dropna(subset=[f for f in features if f in test_df.columns] + [target])
        available_features = [f for f in features if f in test_clean.columns]
        X_test = test_clean[available_features].replace([np.inf, -np.inf], np.nan).fillna(0)
        y_true = test_clean[target].values
        
        # Predict
        y_pred = model.predict(X_test)
        
        # For odds, use average odds from the dataset
        if 'AvgH' in test_clean.columns:
            odds = test_clean['AvgH'].values
        else:
            odds = np.full(len(y_true), 2.0)  # Default odds
        
        # Simulate profit
        profit_sim = simulate_profit(y_true, y_pred, odds)
        
        profit_summary.append({
            'Market': market_name,
            'Total Bets': profit_sim['total_bets'],
            'Win Rate': f"{profit_sim['win_rate']:.1f}%",
            'ROI': f"{profit_sim['roi']:.1f}%",
            'Profit': f"${profit_sim['profit']:.2f}"
        })
    except Exception as e:
        profit_summary.append({
            'Market': market_name,
            'Error': str(e)
        })

profit_df = pd.DataFrame(profit_summary)
print("\nProfit Simulation Results (Historical):")
print("=" * 60)
print(profit_df.to_string(index=False))

## 7. Value Betting Analysis

In [ ]:
# Analyze where model sees value vs bookmakers
if 'match_result' in models:
    model = models['match_result']
    features = model.feature_names
    target = MARKET_TARGETS['match_result']
    
    test_clean = test_df.dropna(subset=[f for f in features if f in test_df.columns] + [target])
    available_features = [f for f in features if f in test_clean.columns]
    X_test = test_clean[available_features].replace([np.inf, -np.inf], np.nan).fillna(0)
    y_true = test_clean[target].values
    
    y_pred = model.predict(X_test)
    y_prob_result = model.predict_proba(X_test)
    y_prob = y_prob_result[0] if isinstance(y_prob_result, tuple) else y_prob_result
    
    # Get max probability for each prediction
    max_probs = np.max(y_prob, axis=1)
    
    # Get odds
    if 'AvgH' in test_clean.columns:
        odds = test_clean['AvgH'].values
    else:
        odds = np.full(len(y_true), 2.0)
    
    # Analyze edge
    analysis, summary = analyze_model_edge(y_true, y_pred, max_probs, odds)
    
    print("Model Edge Analysis:")
    print("=" * 40)
    for k, v in summary.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")

## 8. Walk-Forward Backtest

In [ ]:
# Run walk-forward backtest for match_result
bt_results = walk_forward_backtest(
    df, 'match_result', LEAGUE, 'xgboost',
    train_size=min(1500, len(df) - 200),
    test_size=min(200, max(50, len(df) // 5)),
    step=100
)

if not bt_results.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    bt_results.plot(x='window', y='accuracy', kind='line', ax=ax, marker='o')
    ax.axhline(y=bt_results['accuracy'].mean(), color='red', linestyle='--', 
               label=f'Mean: {bt_results["accuracy"].mean():.4f}')
    ax.set_xlabel('Window')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Walk-Forward Backtest: Match Result ({LEAGUE_CONFIG[LEAGUE]["name"]})')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, '03_walkforward_backtest.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: outputs/03_walkforward_backtest.png")

## 9. Confidence vs Accuracy Analysis

In [ ]:
# Analyze how confidence affects accuracy
if 'match_result' in models:
    model = models['match_result']
    features = model.feature_names
    target = MARKET_TARGETS['match_result']
    
    test_clean = test_df.dropna(subset=[f for f in features if f in test_df.columns] + [target])
    available_features = [f for f in features if f in test_clean.columns]
    X_test = test_clean[available_features].replace([np.inf, -np.inf], np.nan).fillna(0)
    y_true = test_clean[target].values
    
    y_pred = model.predict(X_test)
    y_prob_result = model.predict_proba(X_test)
    y_prob = y_prob_result[0] if isinstance(y_prob_result, tuple) else y_prob_result
    
    max_probs = np.max(y_prob, axis=1)
    correct = (y_pred == y_true)
    
    # Bin by confidence
    bins = [0, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 1.0]
    labels = ['<35%', '35-40%', '40-45%', '45-50%', '50-55%', '55-60%', '60%+']
    
    confidence_bins = pd.cut(max_probs, bins=bins, labels=labels)
    
    analysis_df = pd.DataFrame({
        'confidence': confidence_bins,
        'correct': correct
    })
    
    acc_by_conf = analysis_df.groupby('confidence', observed=True).agg(
        accuracy=('correct', 'mean'),
        count=('correct', 'count')
    )
    
    print("\nAccuracy by Confidence Level:")
    print("=" * 40)
    print(acc_by_conf)
    
    fig, ax1 = plt.subplots(figsize=(10, 5))
    
    acc_by_conf['accuracy'].plot(kind='bar', ax=ax1, color='steelblue', edgecolor='black')
    ax1.set_ylabel('Accuracy', color='steelblue')
    ax1.set_title('Model Accuracy by Confidence Level')
    
    ax2 = ax1.twinx()
    acc_by_conf['count'].plot(kind='line', ax=ax2, color='red', marker='o')
    ax2.set_ylabel('Number of Predictions', color='red')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, '03_confidence_vs_accuracy.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: outputs/03_confidence_vs_accuracy.png")